In [5]:
%load_ext autoreload
%autoreload 2

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

from sklearn.manifold import TSNE, MDS
from torch.utils.data import Subset, DataLoader

import sys
import torch
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")
print(sys.executable)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/home/kathryn/miniconda3/envs/py37/bin/python


In [7]:
from SeqTransformerAE import SeqAE
from data_padded import VAEDataset

### Check compute 

In [22]:
use_cuda = torch.cuda.is_available()
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))
num_cores = !cat /proc/cpuinfo | grep processor | wc -l
num_cores = num_cores[0]
print(num_cores)

1
TITAN Xp
12


### Dataset

In [14]:
data_path = 'data/partial_data.csv'
property_pred = None
dataset = VAEDataset(data_path, property_pred=property_pred, use_cuda=use_cuda)

In [23]:
model = SeqAE(n_tokens = dataset.n_tokens,
             d_model=256,nhead=16,d_latent=64,num_encoder_layers=3,
             num_decoder_layers=3,dim_feedforward=512,dropout=0.1,
             activation='relu',layer_norm_eps=1e-5,batch_first=True)

n_steps = 200 #200
batch_size = 2
lr = 0.00001

sampler = torch.utils.data.RandomSampler(dataset,replacement = False,
                                         num_samples = n_steps*batch_size)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=8, sampler = sampler)

optimizer = torch.optim.Adam(model.parameters(), lr = lr)
scheduler = None

In [28]:
from training_functions import fit
loss_out = fit(model, dataloader, optimizer, scheduler,
               n_epochs = 2, n_steps = n_steps, 
               use_cuda = use_cuda, prop_pred_loss = False,
               recon_alpha = 1., variational = False, use_out_mask = False)

Training in progress...: 100%|██████████| 2/2 [00:07<00:00,  3.95s/it]


In [29]:
torch.cuda.empty_cache()